# PROJECT: CIFAR-10 Image Classifier

Reach for this when you need: 
- Complete end-to-end vision training boilerplate.
- Reference for custom `nn.Module` with residual blocks.
- Standard evaluation logic (Accuracy/Confusion Matrix) for classification.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Data Pipeline

| Step | Action | Logic |
| :--- | :--- | :--- |
| Augmentation | `RandomHorizontalFlip` | Basic regularization |
| Normalization | `Normalize` | (0.5, 0.5, 0.5) for CIFAR-10 |
| Loading | `DataLoader` | batch=128, workers=4, pin=True |

In [ ]:
tform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=tform)
test_ds = datasets.CIFAR10(root='./data', train=False, download=True, transform=tform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

## 2. Model Architecture

Building a custom CNN with Residual-lite blocks as a reference.

In [ ]:
class SimpleResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.BatchNorm2d(ch),
            nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.BatchNorm2d(ch)
        )

    def forward(self, x):
        return torch.relu(x + self.conv(x))

class CIFARClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            SimpleResBlock(32),
            nn.MaxPool2d(2), # 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            SimpleResBlock(64),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model = CIFARClassifier().to(device)
model = torch.compile(model) # Performance boost

## 3. Training Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

def train(model, loader):
    model.train()
    for x, y in tqdm(loader, desc="Training"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

### Key Takeaways
- `torch.compile` provides significant throughput boost for CNNs on modern GPUs.
- `AdaptiveAvgPool2d` makes the model architecture independent of input image size.
- AdamW and weight decay are industry standards for avoiding over-parameterization issues.